# Notebook 3: Fine-tune DistilBERT for Intent Classification

## How to run this
1. Go to **Runtime → Change runtime type → T4 GPU** and click Save
2. Run cells **top to bottom**, one at a time — that's it, **no file uploads needed**
3. The data is embedded directly in Cell 2

Training takes about **10–15 minutes** on the free T4 GPU.

At the end (Cell 9) you will download a zip of the trained model. Unzip it and copy the contents into `quickserve-ai/models/distilbert-intent/` on your computer.

In [ ]:
# Cell 1 — Install packages (run once, ~60 seconds)
!pip install -q transformers[torch] datasets scikit-learn accelerate sentencepiece

In [ ]:
# Cell 2 — Create data directly in Colab (no file upload needed)
import os, json

os.makedirs("data", exist_ok=True)
os.makedirs("models/distilbert-intent", exist_ok=True)
os.makedirs("checkpoints/distilbert-intent", exist_ok=True)
os.makedirs("docs/diagrams", exist_ok=True)

TRAIN = [
    {"text": "hello", "intent": "greeting"},
    {"text": "hi there", "intent": "greeting"},
    {"text": "hey", "intent": "greeting"},
    {"text": "good morning", "intent": "greeting"},
    {"text": "good afternoon", "intent": "greeting"},
    {"text": "hi, how are you", "intent": "greeting"},
    {"text": "hello, I need some help", "intent": "greeting"},
    {"text": "hey there, can you assist me", "intent": "greeting"},
    {"text": "greetings", "intent": "greeting"},
    {"text": "what's up", "intent": "greeting"},
    {"text": "bye", "intent": "goodbye"},
    {"text": "goodbye", "intent": "goodbye"},
    {"text": "see you later", "intent": "goodbye"},
    {"text": "thanks, bye", "intent": "goodbye"},
    {"text": "that's all I needed, bye", "intent": "goodbye"},
    {"text": "thank you, goodbye", "intent": "goodbye"},
    {"text": "have a good day", "intent": "goodbye"},
    {"text": "I'm done for now", "intent": "goodbye"},
    {"text": "take care", "intent": "goodbye"},
    {"text": "catch you later", "intent": "goodbye"},
    {"text": "I want to order 2 pizzas", "intent": "place_order"},
    {"text": "can I order a burger", "intent": "place_order"},
    {"text": "please place an order for 3 sandwiches", "intent": "place_order"},
    {"text": "I'd like to buy 1 coffee", "intent": "place_order"},
    {"text": "order me 2 sodas", "intent": "place_order"},
    {"text": "I want to purchase a laptop bag", "intent": "place_order"},
    {"text": "add 4 notebooks to my cart", "intent": "place_order"},
    {"text": "place order for 1 headphone", "intent": "place_order"},
    {"text": "can you order 5 apples for me", "intent": "place_order"},
    {"text": "I need to order a pair of shoes", "intent": "place_order"},
    {"text": "I want 3 bottles of water delivered", "intent": "place_order"},
    {"text": "order 2 chicken wraps please", "intent": "place_order"},
    {"text": "buy me 1 keyboard", "intent": "place_order"},
    {"text": "get me 6 energy drinks", "intent": "place_order"},
    {"text": "place my order for 2 pens and 1 notebook", "intent": "place_order"},
    {"text": "where is my order", "intent": "track_order"},
    {"text": "track my order ORD123", "intent": "track_order"},
    {"text": "what is the status of order ORD456", "intent": "track_order"},
    {"text": "can you check order ORD789", "intent": "track_order"},
    {"text": "I want to know where my parcel is", "intent": "track_order"},
    {"text": "is my order shipped yet", "intent": "track_order"},
    {"text": "when will my order arrive", "intent": "track_order"},
    {"text": "track order number ORD001", "intent": "track_order"},
    {"text": "what happened to my package", "intent": "track_order"},
    {"text": "check delivery status ORD222", "intent": "track_order"},
    {"text": "has my order been dispatched", "intent": "track_order"},
    {"text": "give me an update on ORD333", "intent": "track_order"},
    {"text": "I'd like to cancel my order", "intent": "cancel_order"},
    {"text": "cancel order ORD123 please", "intent": "cancel_order"},
    {"text": "I don't want the order anymore", "intent": "cancel_order"},
    {"text": "please cancel ORD456", "intent": "cancel_order"},
    {"text": "I want to cancel my last order", "intent": "cancel_order"},
    {"text": "stop my order ORD789", "intent": "cancel_order"},
    {"text": "I changed my mind, cancel it", "intent": "cancel_order"},
    {"text": "delete order ORD001", "intent": "cancel_order"},
    {"text": "I need to cancel order ORD222", "intent": "cancel_order"},
    {"text": "call off my order", "intent": "cancel_order"},
    {"text": "change my order ORD123 to 3 burgers instead", "intent": "modify_order"},
    {"text": "I want to update order ORD456", "intent": "modify_order"},
    {"text": "modify my order please", "intent": "modify_order"},
    {"text": "can I change the quantity in order ORD789", "intent": "modify_order"},
    {"text": "update ORD001 to 2 coffees", "intent": "modify_order"},
    {"text": "I need to edit my order", "intent": "modify_order"},
    {"text": "swap the item in ORD222 to a sandwich", "intent": "modify_order"},
    {"text": "replace the pizza with a burger in ORD333", "intent": "modify_order"},
    {"text": "I'd like to change my order", "intent": "modify_order"},
    {"text": "adjust the quantity in my order", "intent": "modify_order"},
    {"text": "how long does delivery take", "intent": "faq"},
    {"text": "what is your shipping policy", "intent": "faq"},
    {"text": "do you deliver on weekends", "intent": "faq"},
    {"text": "how do I return a product", "intent": "faq"},
    {"text": "what payment methods do you accept", "intent": "faq"},
    {"text": "can I pay with UPI", "intent": "faq"},
    {"text": "is cash on delivery available", "intent": "faq"},
    {"text": "how much is the delivery charge", "intent": "faq"},
    {"text": "what is your return policy", "intent": "faq"},
    {"text": "can I exchange a product", "intent": "faq"},
    {"text": "how do I track a return", "intent": "faq"},
    {"text": "do you offer free shipping", "intent": "faq"},
    {"text": "what are your business hours", "intent": "faq"},
    {"text": "how do I contact support", "intent": "faq"},
    {"text": "where are you located", "intent": "faq"},
    {"text": "I want to talk to a human", "intent": "talk_to_human"},
    {"text": "connect me to an agent", "intent": "talk_to_human"},
    {"text": "I need to speak to someone", "intent": "talk_to_human"},
    {"text": "transfer me to customer support", "intent": "talk_to_human"},
    {"text": "can I talk to a real person", "intent": "talk_to_human"},
    {"text": "get me a support agent", "intent": "talk_to_human"},
    {"text": "I want human assistance", "intent": "talk_to_human"},
    {"text": "escalate this to a person", "intent": "talk_to_human"},
    {"text": "let me speak to someone", "intent": "talk_to_human"},
    {"text": "connect me to your team", "intent": "talk_to_human"},
    {"text": "blah blah blah", "intent": "fallback"},
    {"text": "qwerty uiop", "intent": "fallback"},
    {"text": "what is the meaning of life", "intent": "fallback"},
    {"text": "tell me a joke", "intent": "fallback"},
    {"text": "asdfghjkl", "intent": "fallback"},
    {"text": "can you fly a plane", "intent": "fallback"},
    {"text": "who is the president", "intent": "fallback"},
    {"text": "what is 2 plus 2", "intent": "fallback"},
    {"text": "123 abc xyz", "intent": "fallback"},
    {"text": "random stuff here", "intent": "fallback"},
]

TEST = [
    {"text": "hi", "intent": "greeting"},
    {"text": "good evening", "intent": "greeting"},
    {"text": "hello there", "intent": "greeting"},
    {"text": "farewell", "intent": "goodbye"},
    {"text": "I'm leaving now", "intent": "goodbye"},
    {"text": "order 1 pizza for me", "intent": "place_order"},
    {"text": "I want to buy 2 notebooks", "intent": "place_order"},
    {"text": "get me a coffee please", "intent": "place_order"},
    {"text": "where is order ORD100", "intent": "track_order"},
    {"text": "what's the status of my package", "intent": "track_order"},
    {"text": "cancel ORD100", "intent": "cancel_order"},
    {"text": "I want to call off my order", "intent": "cancel_order"},
    {"text": "update order ORD100 to 1 burger", "intent": "modify_order"},
    {"text": "change quantity in my order", "intent": "modify_order"},
    {"text": "how much does shipping cost", "intent": "faq"},
    {"text": "what are your refund options", "intent": "faq"},
    {"text": "I need a human", "intent": "talk_to_human"},
    {"text": "speak to agent", "intent": "talk_to_human"},
    {"text": "zzz xyz nope", "intent": "fallback"},
    {"text": "what is weather today", "intent": "fallback"},
]

# Write to files
with open("data/intents_train.jsonl", "w") as f:
    for row in TRAIN:
        f.write(json.dumps(row) + "\n")

with open("data/intents_test.jsonl", "w") as f:
    for row in TEST:
        f.write(json.dumps(row) + "\n")

print(f"✅ Train: {len(TRAIN)} examples | Test: {len(TEST)} examples — written to data/")

In [ ]:
# Cell 4 — Load data
def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

train_df = load_jsonl('data/intents_train.jsonl')
test_df  = load_jsonl('data/intents_test.jsonl')

train_df['label'] = train_df['intent'].map(LABEL2ID)
test_df['label']  = test_df['intent'].map(LABEL2ID)

train_ds = Dataset.from_pandas(train_df[['text', 'label']])
test_ds  = Dataset.from_pandas(test_df[['text', 'label']])

print(f'Train: {len(train_ds)} samples | Test: {len(test_ds)} samples')
print('\nIntent distribution (train):')
print(train_df['intent'].value_counts())

In [ ]:
# Cell 5 — Tokenize
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=64)

train_ds = train_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)
train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
print("Tokenization done. Sample:", train_ds[0]['input_ids'][:10])

In [ ]:
# Cell 6 — Load model and train (~10-15 min on T4 GPU)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'macro_f1': f1_score(labels, preds, average='macro')}

args = TrainingArguments(
    output_dir='checkpoints/distilbert-intent',
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=5,
    report_to='none',
    fp16=torch.cuda.is_available(),   # use half-precision on GPU for speed
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Cell 7 — Evaluate and plot results
preds_out = trainer.predict(test_ds)
y_pred = np.argmax(preds_out.predictions, axis=-1)
y_true = test_df['label'].values

macro_f1 = f1_score(y_true, y_pred, average='macro')
print(f"\n✅ DistilBERT Macro F1: {macro_f1:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=LABELS))

# Training loss curve
log_history = trainer.state.log_history
train_loss = [x['loss'] for x in log_history if 'loss' in x]
eval_f1    = [x['eval_macro_f1'] for x in log_history if 'eval_macro_f1' in x]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_loss, color='steelblue')
ax1.set_title('Training Loss — DistilBERT')
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')

ax2.plot(eval_f1, color='darkorange', marker='o')
ax2.set_title('Validation Macro F1 per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Macro F1')

plt.tight_layout()
plt.savefig('docs/diagrams/distilbert_training_curves.png', dpi=150)
plt.show()
print("Saved training curves to docs/diagrams/")

In [ ]:
# Cell 8 — Save model
import os
os.makedirs(MODEL_OUT, exist_ok=True)
model.save_pretrained(MODEL_OUT)
tokenizer.save_pretrained(MODEL_OUT)
print(f"Model saved to {MODEL_OUT}")
print("Files:", os.listdir(MODEL_OUT))

In [ ]:
# Cell 9 — Zip everything and download to your computer
import shutil
from google.colab import files

# Zip the model folder + the training curve image
shutil.make_archive('distilbert_intent_model', 'zip', '.', 'models/distilbert-intent')

# Also include the training curve plot
shutil.copy('docs/diagrams/distilbert_training_curves.png', 'distilbert_training_curves.png')

print("Downloading distilbert_intent_model.zip ...")
files.download('distilbert_intent_model.zip')
files.download('distilbert_training_curves.png')

print()
print("=== NEXT STEPS ===")
print("1. Unzip distilbert_intent_model.zip")
print("2. Copy the contents into:  quickserve-ai/models/distilbert-intent/")
print("3. Copy distilbert_training_curves.png into:  quickserve-ai/docs/diagrams/")
print("4. Mark Milestone 3 'NLP Models Trained' as complete on Qollabb")

In [ ]:
import os
os.makedirs(MODEL_OUT, exist_ok=True)
model.save_pretrained(MODEL_OUT)
tokenizer.save_pretrained(MODEL_OUT)
print(f'Saved model to {MODEL_OUT}')